# E5 · Léxico externo HurtLex
**Parte de:** E0.

**Hipótesis.** El bloque de léxico hecho a mano (H1-H10) fue lo que más aportó en el notebook original (+2,2 puntos). Un léxico externo de palabras ofensivas podría cubrir vocabulario que no aparece en el train.

**Recurso.** *HurtLex* (Bassignana, Basile y Patti, 2018), versión 1.2 en español: 5.006 lemas en 17 categorías. Licencia **CC BY-NC-SA 4.0** (uso no comercial, académico permitido). Fuente: https://github.com/valeriobasile/hurtlex. Copia local en `recursos/hurtlex_ES.tsv`.

**Variables.** Conteo de apariciones por categoría (con plural opcional) + total del nivel *conservative* = 18 columnas nuevas en el bloque de léxico. Los lemas pasan por la misma `normalizar` que los tweets, de modo que *cállate* → `cllate`, igual que la corrupción del corpus.

### Glosario de esta prueba
| Término | Significado |
|---|---|
| **HS / TR / AG** | Las 3 etiquetas: discurso de odio / odio dirigido a una persona concreta / agresividad |
| **F1 medio** | F1 de la clase 1 de cada etiqueta, promediado entre HS, TR y AG (métrica principal) |
| **CV repetida** | Validación cruzada de 5 particiones repetida con 3 semillas (42, 43, 44); se reporta media ± desviación |
| **OOF** | *Out-of-fold*: predicción de cada fila del train hecha por un modelo que no la vio al entrenar |
| **Stacking** | Un metamodelo (regresión logística) aprende a combinar las probabilidades OOF de los modelos base |
| **HurtLex** | Léxico multilingüe de palabras ofensivas, organizado por categorías (animales, prostitución, etnias...) |
| **Lema** | Forma base de una palabra (*puta*, no *putas*) |
| **Nivel conservative** | Subconjunto de HurtLex con las palabras ofensivas más claras |

El glosario completo está en `RESUMEN.md`.

In [1]:
import os, sys, platform
sys.path.insert(0, os.path.abspath('.'))          # el notebook vive en experimentos/, junto a toxic_base.py
import numpy as np, pandas as pd, sklearn, scipy
import toxic_base as tb
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30); pd.set_option('display.max_colwidth', 120)
print('Python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__,
      '| scikit-learn', sklearn.__version__, '| scipy', scipy.__version__)
print('Semillas de la CV repetida:', tb.SEEDS, '| particiones:', tb.N_FOLDS)
d = tb.Datos()
print('Datos en', d.data_dir, '| train', d.train.shape, '| test', d.test.shape)

Python 3.12.3 | numpy 2.5.4 | pandas 3.0.6 | scikit-learn 1.9.1 | scipy 1.18.1
Semillas de la CV repetida: (42, 43, 44) | particiones: 5


Datos en /home/jc/repositories/natural-language-processing-workshops/workshops/Kaggle-tweet-toxic/tweet-toxic-2026-puj | train (1731, 5) | test (1154, 2)


In [2]:
dh = tb.Datos(extra_lex=[tb.hurtlex_variables()])
H = pd.DataFrame(dh.L_TR[:, -18:], columns=dh.lex_nombres[-18:])
pd.DataFrame({'categoría': [tb.HURTLEX_CATEGORIAS.get(c.split('_')[1], 'total conservador') for c in H],
              'tweets con coincidencia': (H > 0).sum().values,
              'P(HS | coincidencia)': [dh.Y[H[c] > 0, 0].mean() for c in H]}, index=H.columns).round(2)

,categoría,tweets con coincidencia,P(HS | coincidencia)
hurtlex_ps,estereotipos étnicos,111,0.46
hurtlex_rci,lugares y gentilicios,2,0.00
hurtlex_pa,profesiones,26,0.35
hurtlex_ddf,discapacidad física,11,0.64
hurtlex_ddp,discapacidad cognitiva,244,0.34
hurtlex_dmc,defectos morales,201,0.42
hurtlex_is,desventaja social,29,0.41
hurtlex_or,plantas,101,0.46
hurtlex_an,animales,1042,0.54
hurtlex_asm,genitales masculinos,241,0.37


Prevalencia de HS en todo el train como referencia: **0,44**.

In [3]:
R = {'E0 · stacking 5 modelos': tb.evaluar(d, {s: tb.stacking(d, tb.BASES_ORIGINALES, s, con_test=False)[0] for s in tb.SEEDS})}
for b in tb.BASES_ORIGINALES:
    R[f'{b} (sin HurtLex)'] = tb.evaluar(d, {s: tb.oof(d, b, s) for s in tb.SEEDS})
    R[f'{b} + HurtLex'] = tb.evaluar(dh, {s: tb.oof(dh, b, s) for s in tb.SEEDS})
R['Stacking + HurtLex'] = tb.evaluar(dh, {s: tb.stacking(dh, tb.BASES_ORIGINALES, s, con_test=False)[0] for s in tb.SEEDS})
tb.tabla(R)

,F1 HS,F1 TR,F1 AG,F1 medio,±,EMR,F1 medio por semilla
Configuración,,,,,,,
E0 · stacking 5 modelos,0.8038,0.8345,0.7994,0.8126,0.0021,0.7757,0.815 / 0.810 / 0.813
CNB (sin HurtLex),0.7509,0.7794,0.7086,0.7463,0.0023,0.7090,0.749 / 0.744 / 0.746
CNB + HurtLex,0.7509,0.7794,0.7086,0.7463,0.0023,0.7090,0.749 / 0.744 / 0.746
LR (sin HurtLex),0.7924,0.8246,0.7899,0.8023,0.0020,0.7499,0.803 / 0.805 / 0.800
LR + HurtLex,0.7928,0.8201,0.7886,0.8005,0.0023,0.7475,0.797 / 0.801 / 0.803
SVM (sin HurtLex),0.7890,0.8279,0.7882,0.8017,0.0009,0.7527,0.803 / 0.801 / 0.801
SVM + HurtLex,0.7864,0.8235,0.7807,0.7969,0.0019,0.7485,0.794 / 0.799 / 0.797
NB-LR (sin HurtLex),0.7916,0.8231,0.7883,0.8010,0.0009,0.7578,0.801 / 0.802 / 0.800
NB-LR + HurtLex,0.7889,0.8160,0.7823,0.7958,0.0004,0.7504,0.796 / 0.795 / 0.796


## Conclusión: **no se adopta** (empeora)
- HurtLex **baja** el F1 medio de todos los modelos (LR 0,802 → 0,801; NB-LR 0,801 → 0,796; stacking 0,813 → 0,809).
- La tabla de arriba explica por qué: en casi todas las categorías, P(HS) cuando hay coincidencia queda cerca de la prevalencia general (0,44). Las categorías más frecuentes (*animales* por *perra*, *prostitución* por *puta*) cubren las palabras clave con las que se **recolectó** el corpus, que aparecen igual en tweets con odio y sin odio.
- Las variables hechas a mano funcionan porque capturan el **contexto** (*hijo* frente a *hija de puta*, *puta* como intensificador), algo que un conteo de léxico no distingue. Las 18 columnas nuevas solo agregan ruido al bloque estandarizado.
- Resultado útil para el informe: en un corpus muestreado por palabras clave, los léxicos genéricos de insultos no discriminan.